# 원하는 포즈로 이미지 만드는 도구 (Pose → Image)

참조 사진의 **자세를 그대로 따라 하는 새 이미지**를 만드는 튜토리얼.

1. 참조 사람 사진에서 **OpenPose**가 관절(스켈레톤)을 뽑는다
2. 그 포즈를 **ControlNet 조건**으로 넣어 **Stable Diffusion 1.5**가 같은 자세의 다른 인물/장면을 만든다

> 무료 Colab **T4 GPU** 에서 동작, **고정 시드(42)** 로 재현됩니다.
> 실행 전: 런타임 > 런타임 유형 변경 > **T4 GPU**

## 0. GPU 확인

In [ ]:
# 이 셀이 하는 일: GPU가 잡혔는지 확인
!nvidia-smi

## 1. 패키지 설치 (몇 분)

In [ ]:
# 이 셀이 하는 일: 생성모델(diffusers) + 포즈추출기(controlnet_aux) 설치
!pip -q install -U diffusers transformers accelerate controlnet_aux mediapipe

## 2. import + 포즈 추출기 로드
Colab의 mediapipe 버전 충돌을 피하려고, 실제로 안 쓰는 mediapipe를 가벼운 껍데기로 대체합니다.

In [ ]:
# 이 셀이 하는 일: mediapipe 호환 우회 + import + OpenPose 추출기 로드
import sys, types
from unittest.mock import MagicMock
_mp = types.ModuleType('mediapipe'); _mp.solutions = MagicMock(); sys.modules['mediapipe'] = _mp

import torch, os, numpy as np
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image
from controlnet_aux import OpenposeDetector

os.makedirs('samples', exist_ok=True)
openpose = OpenposeDetector.from_pretrained('lllyasviel/Annotators')
print('포즈 추출기 준비 완료')

## 3. 참조 사진에서 포즈 뽑기
예시로 인물 사진 URL을 씁니다. **내 사진을 쓰려면** 왼쪽 파일탭에 업로드하고 `load_image('내파일.jpg')` 로 바꾸세요.
(추출 스켈레톤이 비면 준비된 전신 스켈레톤으로 자동 폴백)

In [ ]:
# 이 셀이 하는 일: 참조 사진 → OpenPose로 포즈(스켈레톤) 추출 → 저장
photo = load_image('https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/input_image_vermeer.png')
pose = openpose(photo)
if (np.array(pose.convert('RGB')).sum(2) > 20).mean() < 0.003:
    pose = load_image('https://huggingface.co/datasets/hf-internal-testing/diffusers-images/resolve/main/sd_controlnet/pose.png')
pose.save('samples/pose_01.png')
pose   # 스켈레톤이 자세를 따라그렸는지 확인

## 4. SD1.5 + ControlNet(OpenPose) 파이프라인 로드

In [ ]:
# 이 셀이 하는 일: 포즈 ControlNet + SD1.5 본체를 불러 파이프라인 구성
controlnet = ControlNetModel.from_pretrained('lllyasviel/sd-controlnet-openpose', torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    controlnet=controlnet, torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to('cuda')
print('파이프라인 준비 완료')

## 5. 생성 함수 (고정 시드 = 재현성)
seed 고정 → 같은 입력이면 항상 같은 결과 (루브릭: 재현성).

In [ ]:
# 이 셀이 하는 일: (포즈, 프롬프트) -> 생성 이미지 함수 정의
def generate(pose_img, prompt, seed=42):
    g = torch.Generator('cuda').manual_seed(seed)
    return pipe(prompt, image=pose_img, num_inference_steps=25, generator=g,
                negative_prompt='lowres, bad anatomy, worst quality, blurry').images[0]

## 6. 실행 — 포즈 1 + 프롬프트 A (우주비행사)

In [ ]:
# 이 셀이 하는 일: 같은 포즈에 프롬프트 A 로 생성 → 저장
out1 = generate(pose, 'a photo of an astronaut in a white spacesuit, studio lighting, highly detailed')
out1.save('samples/output_01.png')
out1

## 7. 조건 바꿔보기 — 같은 포즈, 다른 프롬프트 (기사)

In [ ]:
# 이 셀이 하는 일: 같은 포즈에 프롬프트 B 로 생성 → 저장 (자세 유지되는지 관찰)
out2 = generate(pose, 'a medieval knight in shining steel armor, dramatic cinematic lighting, highly detailed')
out2.save('samples/output_02.png')
out2

## 8. 관찰 정리
- **같은 포즈 + 프롬프트만 변경(A→B):** 두 결과 모두 상반신 초상·고개 옆으로 돌린 **같은 자세** 유지, 인물/의상만 우주비행사↔기사로 바뀜 → 입력 포즈가 그대로 반영됨.
- **재현성:** seed=42 고정 → 같은 입력이면 동일 이미지 재현.
- **한계:** 참조가 상반신 초상이라 하반신 포즈는 제어 안 됨(전신은 전신 사진 입력 필요), 손가락 등 미세 부위는 덜 정확.